DOWNLOADING DATASET

In [1]:
!git clone https://github.com/Vidhi1290/Medical-Image-Segmentation-Deep-Learning-Project.git

Cloning into 'Medical-Image-Segmentation-Deep-Learning-Project'...
remote: Enumerating objects: 2520, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 2520 (delta 6), reused 3 (delta 0), pack-reused 2505 (from 1)
Receiving objects: 100% (2520/2520), 156.59 MiB | 52.47 MiB/s, done.
Resolving deltas: 100% (631/631), done.
Updating files: 100% (4933/4933), done.


CHECKING DATASET

In [2]:
import os

base_path = "/content/Medical-Image-Segmentation-Deep-Learning-Project"

for item in os.listdir(base_path):
    print(item)

README.md
data
.git
Code
Medical Image segmentation Notebook
.DS_Store


In [3]:
import os

data_path = "/content/Medical-Image-Segmentation-Deep-Learning-Project/data"

for item in os.listdir(data_path):
    print(item)

PNG
TIF
.DS_Store


In [4]:
import os

tif_path = "/content/Medical-Image-Segmentation-Deep-Learning-Project/data/TIF"
png_path = "/content/Medical-Image-Segmentation-Deep-Learning-Project/data/PNG"

print("TIF files:", os.listdir(tif_path)[:5])
print("PNG files:", os.listdir(png_path)[:5])

TIF files: ['Original', 'Ground Truth', '.DS_Store']
PNG files: ['Original', 'Ground Truth', '.DS_Store']


In [5]:
img_dir = "/content/Medical-Image-Segmentation-Deep-Learning-Project/data/PNG/Original"
mask_dir = "/content/Medical-Image-Segmentation-Deep-Learning-Project/data/PNG/Ground Truth"

SETTING PATHS

In [6]:
img_files = sorted([f for f in os.listdir(img_dir) if not f.startswith('.')])
mask_files = sorted([f for f in os.listdir(mask_dir) if not f.startswith('.')])

print(len(img_files), len(mask_files))

612 612


PREPROCESSING + LOADING

In [7]:
from PIL import Image
import numpy as np
import os

IMG_SIZE = 64

def load_data(img_dir, mask_dir):
    X, y = [], []

    img_files = sorted([f for f in os.listdir(img_dir) if not f.startswith('.')])
    mask_files = sorted([f for f in os.listdir(mask_dir) if not f.startswith('.')])

    for img_file, mask_file in zip(img_files, mask_files):

        # Load grayscale
        img = Image.open(os.path.join(img_dir, img_file)).convert('L')
        mask = Image.open(os.path.join(mask_dir, mask_file)).convert('L')

        # Resize
        img = img.resize((IMG_SIZE, IMG_SIZE))
        mask = mask.resize((IMG_SIZE, IMG_SIZE))

        # Normalize
        img = np.array(img) / 255.0
        mask = np.array(mask) / 255.0

        # 🔥 Binary mask (VERY IMPORTANT)
        mask = (mask > 0.5).astype(np.float32)

        X.append(img)
        y.append(mask)

    X = np.array(X)
    y = np.array(y)

    # Add channel dimension
    X = X[..., np.newaxis]
    y = y[..., np.newaxis]

    return X, y

LOAD DATA

In [8]:
X, y = load_data(img_dir, mask_dir)

In [9]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X range:", X.min(), X.max())
print("Mask values:", np.unique(y))

X shape: (612, 64, 64, 1)
y shape: (612, 64, 64, 1)
X range: 0.00392156862745098 1.0
Mask values: [0. 1.]


CONVERT TO PYTORCH

In [10]:
import torch

X_tensor = torch.from_numpy(X).float().permute(0, 3, 1, 2)
y_tensor = torch.from_numpy(y).float().permute(0, 3, 1, 2)

QML

In [11]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 134.8 MB/s eta 0:00:00


QML LAYER

In [12]:
import torch.nn as nn
import pennylane as qml
from pennylane import numpy as pnp

n_qubits = 8  # Define the number of qubits for the QML layer

# Define the quantum circuit using PennyLane
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

class QMLLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_qubits = n_qubits
        self.weights = nn.Parameter(0.01 * torch.randn(3, n_qubits))

    def forward(self, x):
        outputs = []

        for i in range(x.shape[0]):
            out = quantum_circuit(x[i], self.weights)
            out = torch.stack(out)          # 🔥 keep tensor structure
            outputs.append(out.float())     # 🔥 fix dtype + keep grad

        return torch.stack(outputs)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


UNET + QML

In [13]:
class UNet_QML(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1 = conv_block(1, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = conv_block(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = conv_block(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        # 🔥 Classical bottleneck
        self.bottleneck = conv_block(128, 256)

        # 🔥 QML components
        self.fc1 = nn.Linear(256, n_qubits)
        self.qml = QMLLayer()
        self.fc2 = nn.Linear(n_qubits, 256)

        # Decoder
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = conv_block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = conv_block(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = conv_block(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))  # (B,256,H,W)

        # 🔥 Global pooling → vector
        b_pool = torch.mean(b, dim=(2,3))   # (B,256)

        # 🔥 QML pipeline
        q_in = self.fc1(b_pool)
        q_out = self.qml(q_in)
        q_out = self.fc2(q_out)

        # 🔥 Expand back
        q_out = q_out.unsqueeze(-1).unsqueeze(-1)
        b = b + q_out   # residual fusion

        # Decoder
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)

TRAINING

In [14]:
import torch.nn.functional as F

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    smooth = 1.0

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()


def combined_loss(pred, target):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    dice = dice_loss(pred, target)
    return 0.2 * bce + 1.8 * dice

In [15]:
model = UNet_QML()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
EPOCHS = 100
batch_size = 2   # 🔥 smaller for QML

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for i in range(0, len(X_tensor), batch_size):
        X_batch = X_tensor[i:i+batch_size]
        y_batch = y_tensor[i:i+batch_size]

        optimizer.zero_grad()

        preds = model(X_batch)
        loss = combined_loss(preds, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(X_tensor)//batch_size)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 1.5201
Epoch 2, Loss: 1.4362
Epoch 3, Loss: 1.3681
Epoch 4, Loss: 1.2930
Epoch 5, Loss: 1.1998
Epoch 6, Loss: 1.0953
Epoch 7, Loss: 0.9941
Epoch 8, Loss: 0.8972
Epoch 9, Loss: 0.8284
Epoch 10, Loss: 0.7495
Epoch 11, Loss: 0.6793
Epoch 12, Loss: 0.6453
Epoch 13, Loss: 0.5997
Epoch 14, Loss: 0.5129
Epoch 15, Loss: 0.4800
Epoch 16, Loss: 0.4401
Epoch 17, Loss: 0.3869
Epoch 18, Loss: 0.3696
Epoch 19, Loss: 0.3738
Epoch 20, Loss: 0.3383
Epoch 21, Loss: 0.3126
Epoch 22, Loss: 0.2747
Epoch 23, Loss: 0.2573
Epoch 24, Loss: 0.2439
Epoch 25, Loss: 0.2318
Epoch 26, Loss: 0.2379
Epoch 27, Loss: 0.2542
Epoch 28, Loss: 0.2577
Epoch 29, Loss: 0.2144
Epoch 30, Loss: 0.1980
Epoch 31, Loss: 0.1800
Epoch 32, Loss: 0.1740
Epoch 33, Loss: 0.1587
Epoch 34, Loss: 0.1504
Epoch 35, Loss: 0.1496
Epoch 36, Loss: 0.1566
Epoch 37, Loss: 0.1630
Epoch 38, Loss: 0.1604
Epoch 39, Loss: 0.1445
Epoch 40, Loss: 0.1851
Epoch 41, Loss: 0.1358
Epoch 42, Loss: 0.1181
Epoch 43, Loss: 0.1158
Epoch 44, Loss: 0.10

In [ ]:
import torch
from google.colab import files

# Save model
model_path = "best_model.pth"
torch.save(model.state_dict(), model_path)

# Download to your local system
files.download(model_path)

ACCURACY + METRICS

In [ ]:
import matplotlib.pyplot as plt
import torch

# ---------- METRICS ----------
def dice_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2 * intersection + 1e-7) / (union + 1e-7)).mean().item()

def iou_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1,2,3))
    union = ((pred + target) > 0).float().sum(dim=(1,2,3))
    return ((intersection + 1e-7) / (union + 1e-7)).mean().item()

def pixel_accuracy(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    correct = (pred == target).float().sum()
    total = torch.numel(pred)
    return (correct / total).item()


# ---------- EVALUATION ----------
model.eval()

dice_scores = []
iou_scores = []
acc_scores = []

batch_size = 4

with torch.no_grad():
    for i in range(0, len(X_tensor), batch_size):
        X_batch = X_tensor[i:i+batch_size]
        y_batch = y_tensor[i:i+batch_size]

        preds = model(X_batch)

        dice_scores.append(dice_score(preds, y_batch))
        iou_scores.append(iou_score(preds, y_batch))
        acc_scores.append(pixel_accuracy(preds, y_batch))


# ---------- AVERAGES ----------
avg_dice = sum(dice_scores) / len(dice_scores)
avg_iou = sum(iou_scores) / len(iou_scores)
avg_acc = sum(acc_scores) / len(acc_scores)

print("🔥 FINAL METRICS")
print(f"Dice Score     : {avg_dice:.4f}")
print(f"IoU Score      : {avg_iou:.4f}")
print(f"Pixel Accuracy : {avg_acc:.4f}")


# ---------- GRAPHS ----------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.plot(dice_scores, marker='o')
plt.title("Dice Score per Batch")

plt.subplot(1,3,2)
plt.plot(iou_scores, marker='o')
plt.title("IoU Score per Batch")

plt.subplot(1,3,3)
plt.plot(acc_scores, marker='o')
plt.title("Pixel Accuracy per Batch")

plt.tight_layout()
plt.show()

DISTRIBUTION GRAPH

In [ ]:
plt.hist(dice_scores, bins=10)
plt.title("Dice Score Distribution")
plt.xlabel("Dice")
plt.ylabel("Frequency")
plt.show()

VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

num_samples = 5

plt.figure(figsize=(12, num_samples*3))

for i in range(num_samples):
    idx = np.random.randint(0, len(X_tensor)) # Changed X_val to X_tensor

    input_img = X_tensor[idx].unsqueeze(0)  # Changed X_val to X_tensor
    true_mask = y_tensor[idx].squeeze().numpy() # Changed y_val to y_tensor

    model.eval()
    with torch.no_grad():
        pred = model(input_img)
        pred = torch.sigmoid(pred).squeeze().numpy()

    # 🔥 post-processing
    pred = cv2.GaussianBlur(pred, (3,3), 0)
    pred_mask = (pred > 0.3).astype(np.float32)

    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_CLOSE, kernel)

    img = input_img.squeeze().numpy()

    plt.subplot(num_samples,3,i*3+1)
    plt.imshow(img, cmap='gray')
    plt.title("Image")
    plt.axis('off')

    plt.subplot(num_samples,3,i*3+2)
    plt.imshow(true_mask, cmap='gray')
    plt.title("GT")
    plt.axis('off')

    plt.subplot(num_samples,3,i*3+3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Pred")
    plt.axis('off')

plt.tight_layout()
plt.show()